In [1]:

!pip install pandas numpy plotly openpyxl


In [2]:

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

print("All libraries imported successfully!")


All libraries imported successfully!


In [3]:
import os
print(os.listdir('/content'))

['.config', 'Online Retail.xlsx', 'sample_data']


In [4]:
df = pd.read_excel('/content/Online Retail.xlsx')
print("Dataset loaded successfully!")
print("Shape:", df.shape)
df.head()

Dataset loaded successfully!
Shape: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


**Data Understanding and Cleaning** and **Check the number of rows and columns**

** Check the dataset structure**

In [5]:
# Display first 5 rows
df.head()
print("Dataset Shape:", df.shape)
print("Number of Rows:", df.shape[0])
print("Number of Columns:", df.shape[1])


Dataset Shape: (541909, 8)
Number of Rows: 541909
Number of Columns: 8


**Check data types and information, Check missing values, duplicate records**

In [6]:
df.info()
print("Missing Values in Each Column:")
print(df.isnull().sum())
print("Duplicate Records:", df.duplicated().sum())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB
Missing Values in Each Column:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64
Duplicate Records: 5268


**Clean the dataset**

In [7]:

# Create a copy of the original dataset
df_clean = df.copy()
# Remove records with missing CustomerID
df_clean = df_clean.dropna(subset=['CustomerID'])
# Remove cancelled invoices
df_clean = df_clean[
    ~df_clean['InvoiceNo'].astype(str).str.startswith('C')
]
# Remove invalid quantities
df_clean = df_clean[df_clean['Quantity'] > 0]
# Remove invalid prices
df_clean = df_clean[df_clean['UnitPrice'] > 0]
# Remove duplicate records
df_clean = df_clean.drop_duplicates()
print("Data Cleaning Completed!")
print("Original Dataset Shape:", df.shape)
print("Cleaned Dataset Shape:", df_clean.shape)


Data Cleaning Completed!
Original Dataset Shape: (541909, 8)
Cleaned Dataset Shape: (392692, 8)


**, Create additional date columns**

In [8]:
df_clean['InvoiceDate'] = pd.to_datetime(
    df_clean['InvoiceDate']
)
print("Date conversion completed!")
print(df_clean['InvoiceDate'].dtype)
df_clean['Revenue'] = (
    df_clean['Quantity'] * df_clean['UnitPrice']
)
print("Revenue column created!")
df_clean[['Quantity', 'UnitPrice', 'Revenue']].head()
df_clean['Year'] = df_clean['InvoiceDate'].dt.year
df_clean['Month'] = df_clean['InvoiceDate'].dt.month_name()
df_clean['Month_Number'] = df_clean['InvoiceDate'].dt.month
df_clean['Day'] = df_clean['InvoiceDate'].dt.day_name()
df_clean['Hour'] = df_clean['InvoiceDate'].dt.hour
print("Additional columns created successfully!")
df_clean.head()


Date conversion completed!
datetime64[ns]
Revenue column created!
Additional columns created successfully!


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue,Year,Month,Month_Number,Day,Hour
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,2010,December,12,Wednesday,8
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,December,12,Wednesday,8
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,2010,December,12,Wednesday,8
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,December,12,Wednesday,8
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,December,12,Wednesday,8


**Generate a final data summary**

In [9]:
print("FINAL DATASET SUMMARY")
print("----------------------")
print("Total Transactions:", df_clean['InvoiceNo'].nunique())
print("Total Products:", df_clean['StockCode'].nunique())
print("Total Customers:", df_clean['CustomerID'].nunique())
print("Total Countries:", df_clean['Country'].nunique())
print("Total Quantity Sold:", df_clean['Quantity'].sum())
print("Total Revenue:", round(df_clean['Revenue'].sum(), 2))


FINAL DATASET SUMMARY
----------------------
Total Transactions: 18532
Total Products: 3665
Total Customers: 4338
Total Countries: 37
Total Quantity Sold: 5152002
Total Revenue: 8887208.89


**Data Visualisation**

**Set a consistent dashboard theme**

In [10]:

import plotly.express as px
import plotly.graph_objects as go
# Set common dashboard theme
pio.templates.default = "plotly_white"
# Consistent color palette
colors = px.colors.qualitative.Set2
print("Dashboard theme configured successfully!")


Dashboard theme configured successfully!


**Chart 1 – Monthly Revenue Trend (Line Chart)**

In [11]:
# Group revenue by month
monthly_sales = (
    df_clean.groupby(['Year', 'Month_Number', 'Month'])['Revenue']
    .sum()
    .reset_index()
    .sort_values(['Year', 'Month_Number'])
)
# Create line chart
fig1 = px.line(
    monthly_sales,
    x='Month',
    y='Revenue',
    color='Year',
    markers=True,
    title='Monthly Revenue Trend',
    labels={
        'Month': 'Month',
        'Revenue': 'Total Revenue',
        'Year': 'Year'
    },
    template='plotly_white'
)
fig1.update_layout(
    title_x=0.5,
    font=dict(size=13),
    hovermode='x unified'
)
fig1.show()


**Chart 2 – Top 10 Products by Revenue (Bar Chart)**

In [12]:

# Group revenue by product
top_products = (
    df_clean.groupby('Description')['Revenue']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
# Create bar chart
fig2 = px.bar(
    top_products,
    x='Revenue',
    y='Description',
    orientation='h',
    title='Top 10 Products by Revenue',
    color='Revenue',
    color_continuous_scale='Blues',
    template='plotly_white',
    labels={
        'Revenue': 'Total Revenue',
        'Description': 'Product'
    }
)
fig2.update_layout(
    title_x=0.5,
    yaxis={'categoryorder': 'total ascending'}
)

fig2.show()


**Chart 3 – Country-wise Revenue (Pie Chart)**

In [13]:

# Calculate revenue by country
country_sales = (
    df_clean.groupby('Country')['Revenue']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

# Create pie chart
fig3 = px.pie(
    country_sales,
    names='Country',
    values='Revenue',
    title='Top 10 Countries by Revenue',
    hole=0.35,
    template='plotly_white'
)

fig3.update_traces(
    textposition='inside',
    textinfo='percent+label',
    hovertemplate=(
        '<b>%{label}</b><br>'
        'Revenue: £%{value:,.2f}<br>'
        'Share: %{percent}<extra></extra>'
    )
)

fig3.update_layout(title_x=0.5)

fig3.show()


**Chart 4 – Product Price Distribution (Histogram)**

In [14]:

fig4 = px.histogram(
    df_clean,
    x='UnitPrice',
    nbins=50,
    title='Product Price Distribution',
    color_discrete_sequence=['#3498DB'],
    labels={
        'UnitPrice': 'Unit Price (£)',
        'count': 'Number of Transactions'
    },
    template='plotly_white'
)

fig4.update_layout(
    title_x=0.5,
    xaxis_title='Unit Price (£)',
    yaxis_title='Frequency'
)

fig4.show()


**Chart 5 – Quantity vs Revenue (Scatter Plot)**

In [15]:

# Sample records to make the chart faster
scatter_data = df_clean.sample(
    n=min(3000, len(df_clean)),
    random_state=42
)

fig5 = px.scatter(
    scatter_data,
    x='Quantity',
    y='Revenue',
    color='Country',
    hover_data=['Description', 'UnitPrice', 'InvoiceNo'],
    title='Quantity vs Revenue',
    opacity=0.6,
    template='plotly_white'
)

fig5.update_layout(
    title_x=0.5,
    xaxis_title='Quantity Sold',
    yaxis_title='Revenue (£)'
)

fig5.show()


**Chart 6 – Sales Activity (Heatmap)**

In [16]:

# Create day and month sales summary
heatmap_data = (
    df_clean.pivot_table(
        index='Day',
        columns='Month_Number',
        values='Revenue',
        aggfunc='sum',
        fill_value=0
    )
)

# Arrange weekdays correctly
day_order = [
    'Monday', 'Tuesday', 'Wednesday',
    'Thursday', 'Friday', 'Saturday', 'Sunday'
]

heatmap_data = heatmap_data.reindex(day_order)

fig6 = px.imshow(
    heatmap_data,
    labels={
        'x': 'Month Number',
        'y': 'Day of Week',
        'color': 'Revenue (£)'
    },
    x=heatmap_data.columns,
    y=heatmap_data.index,
    color_continuous_scale='Blues',
    title='Sales Activity by Day and Month',
    aspect='auto',
    template='plotly_white'
)

fig6.update_layout(title_x=0.5)

fig6.show()


**Create the dashboard**

In [19]:

# ============================================================
# ONLINE RETAIL SALES ANALYSIS - INTERACTIVE DASHBOARD
# ============================================================

!pip install -q pandas numpy plotly openpyxl ipywidgets

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets

from plotly.subplots import make_subplots
from IPython.display import display, HTML
from google.colab import output

output.enable_custom_widget_manager()

# ============================================================
# 1. LOAD AND CLEAN DATA
# ============================================================

df = pd.read_excel('/content/Online Retail.xlsx')

df_clean = df.copy()

df_clean = df_clean.dropna(subset=['CustomerID'])
df_clean = df_clean[
    ~df_clean['InvoiceNo'].astype(str).str.startswith('C')
]
df_clean = df_clean[df_clean['Quantity'] > 0]
df_clean = df_clean[df_clean['UnitPrice'] > 0]
df_clean = df_clean.drop_duplicates()

df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
df_clean['Revenue'] = df_clean['Quantity'] * df_clean['UnitPrice']

df_clean['Year'] = df_clean['InvoiceDate'].dt.year
df_clean['Month'] = df_clean['InvoiceDate'].dt.month
df_clean['Day'] = df_clean['InvoiceDate'].dt.day_name()

print("Dataset loaded and cleaned successfully!")
print("Rows:", len(df_clean))

# ============================================================
# 2. DASHBOARD TITLE
# ============================================================

display(HTML("""
<div style="
    background: linear-gradient(120deg, #14213d, #264653);
    padding: 25px;
    border-radius: 12px;
    text-align: center;
    color: white;
    margin-bottom: 18px;
">
    <h1 style="margin:0; font-size:30px;">
        ONLINE RETAIL SALES ANALYSIS
    </h1>
    <p style="margin:10px 0 0; font-size:15px;">
        Interactive Dashboard | Sales Trends | Customer Behaviour |
        Product Performance
    </p>
</div>
"""))

# ============================================================
# 3. INTERACTIVE FILTERS
# ============================================================

years = sorted(df_clean['Year'].dropna().unique())

countries = ['All Countries'] + sorted(
    df_clean['Country'].dropna().unique().tolist()
)

year_filter = widgets.Dropdown(
    options=['All Years'] + years,
    value='All Years',
    description='Year:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='220px')
)

country_filter = widgets.Dropdown(
    options=countries,
    value='All Countries',
    description='Country:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

display(HTML("<h3 style='color:#264653;'>Dashboard Filters</h3>"))
display(widgets.HBox([year_filter, country_filter]))

# ============================================================
# 4. DASHBOARD FUNCTION
# ============================================================

def create_dashboard(selected_year, selected_country):

    data = df_clean.copy()

    # Apply year filter
    if selected_year != 'All Years':
        data = data[data['Year'] == selected_year]

    # Apply country filter
    if selected_country != 'All Countries':
        data = data[data['Country'] == selected_country]

    if data.empty:
        print("No data available for these filters.")
        return

    # --------------------------------------------------------
    # KPI CALCULATIONS
    # --------------------------------------------------------

    total_revenue = data['Revenue'].sum()
    total_customers = data['CustomerID'].nunique()
    total_products = data['StockCode'].nunique()
    total_transactions = data['InvoiceNo'].nunique()

    # --------------------------------------------------------
    # CHART 1: MONTHLY REVENUE TREND
    # --------------------------------------------------------

    monthly_data = (
        data.set_index('InvoiceDate')
        .resample('MS')['Revenue']
        .sum()
        .reset_index()
    )

    # --------------------------------------------------------
    # CHART 2: TOP 10 PRODUCTS
    # --------------------------------------------------------

    product_data = (
        data.groupby('Description')['Revenue']
        .sum()
        .nlargest(10)
        .sort_values()
        .reset_index()
    )

    # --------------------------------------------------------
    # CHART 3: TOP 10 COUNTRIES
    # --------------------------------------------------------

    country_data = (
        data.groupby('Country')['Revenue']
        .sum()
        .nlargest(10)
        .reset_index()
    )

    # --------------------------------------------------------
    # CHART 4: UNIT PRICE DISTRIBUTION
    # --------------------------------------------------------

    price_limit = data['UnitPrice'].quantile(0.99)

    price_data = data[
        data['UnitPrice'] <= price_limit
    ]

    # --------------------------------------------------------
    # CHART 5: QUANTITY VS REVENUE
    # --------------------------------------------------------

    scatter_data = data.sample(
        n=min(3000, len(data)),
        random_state=42
    )

    # --------------------------------------------------------
    # CHART 6: SALES BY DAY AND MONTH
    # --------------------------------------------------------

    day_order = [
        'Monday', 'Tuesday', 'Wednesday',
        'Thursday', 'Friday', 'Saturday', 'Sunday'
    ]

    heatmap_data = data.pivot_table(
        index='Day',
        columns='Month',
        values='Revenue',
        aggfunc='sum',
        fill_value=0
    )

    heatmap_data = heatmap_data.reindex(day_order)

    # --------------------------------------------------------
    # CREATE SUBPLOTS
    # --------------------------------------------------------

    fig = make_subplots(
        rows=3,
        cols=2,

        subplot_titles=(
            "Monthly Revenue Trend",
            "Top 10 Products by Revenue",
            "Revenue Contribution by Country",
            "Unit Price Distribution",
            "Quantity vs Revenue",
            "Sales by Day and Month"
        ),

        specs=[
            [{"type": "xy"}, {"type": "xy"}],
            [{"type": "domain"}, {"type": "xy"}],
            [{"type": "xy"}, {"type": "heatmap"}]
        ],

        vertical_spacing=0.15,
        horizontal_spacing=0.13
    )

    # --------------------------------------------------------
    # ADD CHART 1: LINE CHART
    # --------------------------------------------------------

    fig.add_trace(
        go.Scatter(
            x=monthly_data['InvoiceDate'],
            y=monthly_data['Revenue'],
            mode='lines+markers',
            name='Monthly Revenue',
            line=dict(color='#264653', width=3),
            marker=dict(size=7),
            fill='tozeroy',
            fillcolor='rgba(42,157,143,0.12)',
            hovertemplate='Date: %{x|%b %Y}<br>Revenue: £%{y:,.2f}<extra></extra>'
        ),
        row=1, col=1
    )

    # --------------------------------------------------------
    # ADD CHART 2: BAR CHART
    # --------------------------------------------------------

    fig.add_trace(
        go.Bar(
            x=product_data['Revenue'],
            y=product_data['Description'],
            orientation='h',
            marker_color='#2a9d8f',
            name='Products',
            hovertemplate='%{y}<br>Revenue: £%{x:,.2f}<extra></extra>'
        ),
        row=1, col=2
    )

    # --------------------------------------------------------
    # ADD CHART 3: PIE CHART
    # --------------------------------------------------------

    fig.add_trace(
        go.Pie(
            labels=country_data['Country'],
            values=country_data['Revenue'],
            hole=0.45,
            name='Countries',
            textinfo='percent',
            hovertemplate='%{label}<br>Revenue: £%{value:,.2f}<br>Share: %{percent}<extra></extra>'
        ),
        row=2, col=1
    )

    # --------------------------------------------------------
    # ADD CHART 4: HISTOGRAM
    # --------------------------------------------------------

    fig.add_trace(
        go.Histogram(
            x=price_data['UnitPrice'],
            nbinsx=35,
            marker_color='#e9c46a',
            name='Unit Price',
            hovertemplate='Unit Price: £%{x}<br>Count: %{y}<extra></extra>'
        ),
        row=2, col=2
    )

    # --------------------------------------------------------
    # ADD CHART 5: SCATTER PLOT
    # --------------------------------------------------------

    fig.add_trace(
        go.Scattergl(
            x=scatter_data['Quantity'],
            y=scatter_data['Revenue'],
            mode='markers',
            marker=dict(
                color=scatter_data['UnitPrice'],
                colorscale='Viridis',
                size=6,
                opacity=0.65,
                colorbar=dict(title='Unit Price')
            ),
            text=scatter_data['Description'],
            customdata=scatter_data[['UnitPrice', 'InvoiceNo']],
            name='Transactions',
            hovertemplate=(
                'Product: %{text}<br>'
                'Quantity: %{x}<br>'
                'Revenue: £%{y:,.2f}<br>'
                'Unit Price: £%{customdata[0]:.2f}<br>'
                'Invoice: %{customdata[1]}'
                '<extra></extra>'
            )
        ),
        row=3, col=1
    )

    # --------------------------------------------------------
    # ADD CHART 6: HEATMAP
    # --------------------------------------------------------

    fig.add_trace(
        go.Heatmap(
            z=heatmap_data.values,
            x=heatmap_data.columns,
            y=heatmap_data.index,
            colorscale='YlGnBu',
            colorbar=dict(title='Revenue'),
            hovertemplate='Month: %{x}<br>Day: %{y}<br>Revenue: £%{z:,.2f}<extra></extra>'
        ),
        row=3, col=2
    )

    # --------------------------------------------------------
    # KPI CARDS - SEPARATE FROM TITLE
    # --------------------------------------------------------

    kpis = [
        ("TOTAL REVENUE", f"£{total_revenue:,.2f}", "#264653"),
        ("CUSTOMERS", f"{total_customers:,}", "#2a9d8f"),
        ("PRODUCTS", f"{total_products:,}", "#e9a23b"),
        ("TRANSACTIONS", f"{total_transactions:,}", "#457b9d")
    ]

    for i, (label, value, color) in enumerate(kpis):
        x_position = 0.125 + i * 0.25

        fig.add_annotation(
            x=x_position,
            y=1.15,
            xref='paper',
            yref='paper',
            text=f"<b>{label}</b><br><span style='font-size:21px'>{value}</span>",
            showarrow=False,
            align='center',
            bgcolor=color,
            bordercolor=color,
            borderwidth=1,
            borderpad=12,
            font=dict(color='white', size=13)
        )

    # --------------------------------------------------------
    # FINAL DASHBOARD DESIGN
    # --------------------------------------------------------

    fig.update_layout(
        title=dict(
            text="ONLINE RETAIL SALES DASHBOARD",
            x=0.5,
            y=0.98,
            xanchor='center',
            font=dict(size=24, color='#14213d')
        ),

        height=1550,
        width=1500,

        margin=dict(
            t=300,
            b=100,
            l=100,
            r=120
        ),

        template='plotly_white',

        paper_bgcolor='#f5f7fa',
        plot_bgcolor='white',

        font=dict(
            family='Arial',
            size=11,
            color='#263238'
        ),

        hovermode='closest',
        showlegend=False
    )

    # Axis labels
    fig.update_xaxes(title_text="Date", row=1, col=1)
    fig.update_yaxes(title_text="Revenue (£)", row=1, col=1)

    fig.update_xaxes(title_text="Revenue (£)", row=1, col=2)
    fig.update_yaxes(title_text="Product", row=1, col=2)

    fig.update_xaxes(title_text="Unit Price (£)", row=2, col=2)
    fig.update_yaxes(title_text="Frequency", row=2, col=2)

    fig.update_xaxes(title_text="Quantity Sold", row=3, col=1)
    fig.update_yaxes(title_text="Revenue (£)", row=3, col=1)

    fig.update_xaxes(title_text="Month", row=3, col=2)
    fig.update_yaxes(title_text="Day of Week", row=3, col=2)

    fig.show(config={
        'displaylogo': False,
        'scrollZoom': True,
        'responsive': True
    })


# ============================================================
# 5. CONNECT FILTERS TO DASHBOARD
# ============================================================

dashboard_output = widgets.interactive_output(
    create_dashboard,
    {
        'selected_year': year_filter,
        'selected_country': country_filter
    }
)

display(dashboard_output)

# ============================================================
# END OF INTERACTIVE DASHBOARD
# ============================================================


Dataset loaded and cleaned successfully!
Rows: 392692


Output()

In [20]:

# ============================================================
# EXPORT INTERACTIVE DASHBOARD AS HTML
# ============================================================

# Recreate the dashboard using the current filter selections
selected_year = year_filter.value
selected_country = country_filter.value

# Prepare the filtered dataset
data = df_clean.copy()

if selected_year != 'All Years':
    data = data[data['Year'] == selected_year]

if selected_country != 'All Countries':
    data = data[data['Country'] == selected_country]

# Create a standalone Plotly HTML dashboard
monthly_data = (
    data.set_index('InvoiceDate')
    .resample('MS')['Revenue']
    .sum()
    .reset_index()
)

product_data = (
    data.groupby('Description')['Revenue']
    .sum()
    .nlargest(10)
    .sort_values()
    .reset_index()
)

country_data = (
    data.groupby('Country')['Revenue']
    .sum()
    .nlargest(10)
    .reset_index()
)

price_limit = data['UnitPrice'].quantile(0.99)
price_data = data[data['UnitPrice'] <= price_limit]

scatter_data = data.sample(
    n=min(3000, len(data)),
    random_state=42
)

day_order = [
    'Monday', 'Tuesday', 'Wednesday',
    'Thursday', 'Friday', 'Saturday', 'Sunday'
]

heatmap_data = data.pivot_table(
    index='Day',
    columns='Month',
    values='Revenue',
    aggfunc='sum',
    fill_value=0
).reindex(day_order)

# Create dashboard layout
export_fig = make_subplots(
    rows=3,
    cols=2,
    subplot_titles=(
        "Monthly Revenue Trend",
        "Top 10 Products by Revenue",
        "Revenue by Country",
        "Unit Price Distribution",
        "Quantity vs Revenue",
        "Sales by Day and Month"
    ),
    specs=[
        [{"type": "xy"}, {"type": "xy"}],
        [{"type": "domain"}, {"type": "xy"}],
        [{"type": "xy"}, {"type": "heatmap"}]
    ],
    vertical_spacing=0.15,
    horizontal_spacing=0.13
)

# Chart 1: Line
export_fig.add_trace(
    go.Scatter(
        x=monthly_data['InvoiceDate'],
        y=monthly_data['Revenue'],
        mode='lines+markers',
        name='Monthly Revenue',
        line=dict(color='#264653', width=3)
    ),
    row=1, col=1
)

# Chart 2: Bar
export_fig.add_trace(
    go.Bar(
        x=product_data['Revenue'],
        y=product_data['Description'],
        orientation='h',
        marker_color='#2a9d8f'
    ),
    row=1, col=2
)

# Chart 3: Pie
export_fig.add_trace(
    go.Pie(
        labels=country_data['Country'],
        values=country_data['Revenue'],
        hole=0.45
    ),
    row=2, col=1
)

# Chart 4: Histogram
export_fig.add_trace(
    go.Histogram(
        x=price_data['UnitPrice'],
        nbinsx=35,
        marker_color='#e9c46a'
    ),
    row=2, col=2
)

# Chart 5: Scatter
export_fig.add_trace(
    go.Scattergl(
        x=scatter_data['Quantity'],
        y=scatter_data['Revenue'],
        mode='markers',
        marker=dict(
            color=scatter_data['UnitPrice'],
            colorscale='Viridis',
            size=6
        ),
        text=scatter_data['Description'],
        name='Transactions'
    ),
    row=3, col=1
)

# Chart 6: Heatmap
export_fig.add_trace(
    go.Heatmap(
        z=heatmap_data.values,
        x=heatmap_data.columns,
        y=heatmap_data.index,
        colorscale='YlGnBu'
    ),
    row=3, col=2
)

# Dashboard title and KPI cards
kpis = [
    ("TOTAL REVENUE", f"£{data['Revenue'].sum():,.2f}", "#264653"),
    ("CUSTOMERS", f"{data['CustomerID'].nunique():,}", "#2a9d8f"),
    ("PRODUCTS", f"{data['StockCode'].nunique():,}", "#e9a23b"),
    ("TRANSACTIONS", f"{data['InvoiceNo'].nunique():,}", "#457b9d")
]

for i, (label, value, color) in enumerate(kpis):
    export_fig.add_annotation(
        x=0.125 + i * 0.25,
        y=1.15,
        xref='paper',
        yref='paper',
        text=f"<b>{label}</b><br><span style='font-size:21px'>{value}</span>",
        showarrow=False,
        bgcolor=color,
        bordercolor=color,
        borderpad=12,
        font=dict(color='white', size=13)
    )

export_fig.update_layout(
    title=dict(
        text="ONLINE RETAIL SALES ANALYSIS DASHBOARD",
        x=0.5,
        y=0.98,
        xanchor='center',
        font=dict(size=24, color='#14213d')
    ),
    height=1550,
    width=1500,
    margin=dict(t=300, b=100, l=100, r=120),
    template='plotly_white',
    paper_bgcolor='#f5f7fa',
    plot_bgcolor='white',
    hovermode='closest',
    showlegend=False
)

# Save HTML file
export_fig.write_html(
    '/content/Online_Retail_Dashboard.html',
    include_plotlyjs=True,
    full_html=True,
    config={
        'displaylogo': False,
        'scrollZoom': True,
        'responsive': True
    }
)

print("SUCCESS! Dashboard exported as HTML.")
print("File: Online_Retail_Dashboard.html")


SUCCESS! Dashboard exported as HTML.
File: Online_Retail_Dashboard.html


In [21]:
import os

print(os.path.exists('/content/Online_Retail_Dashboard.html'))

True


In [22]:
from google.colab import files

files.download('/content/Online_Retail_Dashboard.html')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>